# Sinh 60 tuyến theo LEZ 2027/2030
A = thí điểm LEZ 2027 (KV1/KV2/KV3); B = LEZ 2030 trừ A; C = Hà Nội ngoài LEZ 2030. Sinh **20 A/20 B/20 C**. Giữ thuật toán ma trận sinh tuyến, graph có hướng và tiêu chí RC hiện có. Dân cư/hoạt động không dịch chuyển ranh giới LEZ.
Python thường gọi worker PyQGIS đã đăng ký; không gửi HTTP. Tài liệu: `work/docs/lez/SAMPLING_ABC.md`.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [p / "work" for p in candidates]
candidates += [Path.home() / "python" / "work"]
WORK = next((p.resolve() for p in candidates if (p / "scripts/lez/project_automation.py").is_file()), None)
if WORK is None:
    raise FileNotFoundError("Mở notebook từ folder work có scripts/lez/project_automation.py.")
if str(WORK / "scripts") not in sys.path:
    sys.path.insert(0, str(WORK / "scripts"))
from lez.common import input_path

from lez.sampling_abc import preflight, run_sampling, status

## 1. Cấu hình và nguồn
Ranh giới theo `work/config/lez_policy.json`. Hình học OSM là proxy kỹ thuật, chưa phải GIS chính thức. Chuẩn hóa V/N cũ chỉ là nguồn Hà Nội và lớp đối chiếu lịch sử; notebook này không chạy lại điều chỉnh theo dân cư.

In [ ]:
OPTIONS = json.loads((WORK / "config/sampling_abc.json").read_text())
POLICY = json.loads((WORK / "config/lez_policy.json").read_text())
assert OPTIONS["policy_version"] == POLICY["policy_version"]
assert not POLICY["allow_population_activity_boundary_adjustments"]
display(pd.DataFrame(OPTIONS["targets"])[["route_id", "sampling_zone", "rc_target", "group", "policy_version"]])
print(json.dumps(OPTIONS.get("target_reallocation"), ensure_ascii=False, indent=2))

## 2. Kiểm tra trước khi chạy
Kiểm tra nguồn, snapshot, hash và runtime; không gửi API. **A×RC1 được ghi nhận là giới hạn của thiết kế mẫu hiện tại**: có khoảng 5,707 km mạng RC1 giao với A nhưng còn 0 m đủ sinh tuyến. Trong 108 phần giao với A, 3 phần vượt ranh giới và 105 phần nằm trọn A bị loại vì giao dải bất định.

Giữ ô và mẫu số trong ma trận/D; báo 11/12 ô có tuyến. 20 mục tiêu A chia RC2/RC3/RC4 theo 6/7/7. Xem [giải thích A×RC1](../docs/lez/A_RC1_LIMITATION.md) và [bằng chứng](../reports/sampling/A_RC1_limitation.json). Khi đầu vào hoặc dải bất định thay đổi có căn cứ, đánh giá lại trước khi kết luận ô đã khả dụng.


In [ ]:
CHECK = preflight(WORK)
assert CHECK["network_calls"] == 0
print(json.dumps(CHECK, ensure_ascii=False, indent=2))

## 3. Chạy hoặc tiếp tục
`force=False` dùng cache đúng fingerprint và hash. Sau Ctrl+C chạy lại cell này. Không cần tải OSM lại.

In [ ]:
RESULT = run_sampling(WORK, force=False, progress=print)
print(json.dumps(RESULT["candidates"], ensure_ascii=False, indent=2))

## 4. Tiến độ và đầu ra hiện hành
Đếm tuyến từ CSV, không dùng số mục tiêu thay cho kết quả. Báo cáo mới nhất xác định bundle đang dùng.

In [ ]:
RESULT = status(WORK)
DIRECTORY = input_path(WORK, RESULT["paths"]["directory"])
CANDIDATES = pd.read_csv(DIRECTORY / "candidate_routes.csv")
display(CANDIDATES)
display(CANDIDATES.groupby(["sampling_zone", "group"]).size().rename("route_count").reset_index())
print("Mục tiêu chưa sinh:", RESULT["candidates"].get("failed_targets", []))

## 5. Cơ cấu RC, độ phủ và hình học
Chiều dài mạng đếm phần vật lý một lần; chiều dài tập tuyến cộng lượt đi RC. D đo sai lệch cơ cấu chiều dài. Độ phủ duy nhất dùng phần RC đã đi ít nhất một lần chia chiều dài mạng RC. Connector báo riêng, không trộn vào mẫu số RC.

In [ ]:
for name in ("network_matrix.csv", "sample_matrix.csv", "connector_diagnostic.csv"):
    path = DIRECTORY / name
    if path.is_file():
        print(name)
        display(pd.read_csv(path))
print(json.dumps(RESULT.get("composition"), ensure_ascii=False, indent=2))
print("Nguồn ranh giới:", DIRECTORY / "policy_boundaries_report.json")

## 6. Chọn bộ 30
Mở `work/notebooks/06_select_30_routes.ipynb`: chọn **10 A + 8 B + 12 C**, tương ứng **18 trong/12 ngoài LEZ 2030**. `work/notebooks/09_automate_project.ipynb` chạy toàn bộ quy trình hiện hành, đến kiểm tra chuyến đo; phần xây chu trình lái được loại khỏi lần tự động hóa này.